In [ ]:
from pathlib import Path

import uproot

DATA_FILENAME = "JetNtuple_RunIISummer16_13TeV_MC_1.root"
SEARCH_ROOTS = (Path.cwd(), Path.cwd().parent)
candidate_paths = [
    base / relative_path
    for base in SEARCH_ROOTS
    for relative_path in (
        Path("data") / "raw" / DATA_FILENAME,
        Path("QCDJetsMachineLearning") / DATA_FILENAME,
    )
]
ROOT_PATH = next((path for path in candidate_paths if path.is_file()), None)
if ROOT_PATH is None:
    raise FileNotFoundError(f"Could not find {DATA_FILENAME} from {Path.cwd()}")
ROOT_PATH = ROOT_PATH.resolve()
print(f"Data file: {ROOT_PATH}")

f = uproot.open(ROOT_PATH)
classnames = f.classnames(recursive=True)   # {name: ROOT classname} — digs into every subdirectory, not just the top level

# takes objects of a certain type (Tree or RNTuple)
tree_candidates = [k for k, cls in classnames.items() if "Tree" in cls or "RNTuple" in cls]

# function to extract the cycle number (version)
def cycle_num(key):
    return int(key.rsplit(";", 1)[1]) if ";" in key else 0

# the goal is to take the highest cycle (newest version)
tree_candidates.sort(key=cycle_num, reverse=True)   
tree = f[tree_candidates[0]]

print(f"Using: {tree_candidates[0]}")
#print(tree.keys())   # branch/features names

for key in tree.keys():
    print(key)

In [ ]:
import awkward as ak
import numpy as np
import torch
from torch_geometric.data import Data

K_NEIGHBORS = 8
particle_feature_names = ("pT", "dEta", "dPhi")

# Read labels, event IDs, particle features, and the AK4 constituent mask.
branches = [
    "event",
    "physFlav",
    "isPhysUDS",
    "isPhysG",
    "PF_pT",
    "PF_dEta",
    "PF_dPhi",
    "PF_fromAK4Jet",
]
raw_data = tree.arrays(branches, library="ak")

# Keep only jets with one unambiguous binary label.
is_quark = raw_data["isPhysUDS"] == 1
is_gluon = raw_data["isPhysG"] == 1
selected_jets = raw_data[is_quark ^ is_gluon]


def build_knn_edges(pos, k=K_NEIGHBORS):
    """Return directed k-nearest-neighbour edges using native PyTorch."""
    num_nodes = pos.size(0)
    if num_nodes < 2:
        return torch.empty((2, 0), dtype=torch.long)

    k = min(k, num_nodes - 1)
    distances = torch.cdist(pos, pos)
    distances.fill_diagonal_(float("inf"))
    neighbours = distances.topk(k, largest=False).indices

    target = torch.arange(num_nodes).repeat_interleave(k)
    source = neighbours.reshape(-1)
    return torch.stack((source, target), dim=0)


# Create one PyTorch Geometric Data object per jet.
jet_graphs = []
for jet in selected_jets:
    constituent_mask = ak.to_numpy(jet["PF_fromAK4Jet"]) == 1
    particle_matrix = np.column_stack(
        [
            ak.to_numpy(jet["PF_pT"])[constituent_mask],
            ak.to_numpy(jet["PF_dEta"])[constituent_mask],
            ak.to_numpy(jet["PF_dPhi"])[constituent_mask],
        ]
    )

    x = torch.as_tensor(particle_matrix, dtype=torch.float32)
    pos = x[:, 1:3].clone()

    jet_graphs.append(
        Data(
            x=x,
            pos=pos,
            edge_index=build_knn_edges(pos),
            y=torch.tensor([int(jet["isPhysG"])], dtype=torch.long),
            original_flavor=torch.tensor([int(jet["physFlav"])], dtype=torch.long),
            event_id=torch.tensor([int(jet["event"])], dtype=torch.long),
        )
    )

print(f"PyG graphs created: {len(jet_graphs):,} / {len(raw_data):,} jets")
print("Node feature columns:", particle_feature_names)
print("Target convention: 0 = light quark (UDS), 1 = gluon")
print(jet_graphs[0])

# PyTorch Geometric jet graphs

The previous cell creates one `torch_geometric.data.Data` object per selected jet and stores the complete collection in `jet_graphs`.

Each graph contains:

- `x`: particle node features `[pT, dEta, dPhi]` with shape `[num_particles, 3]`.
- `pos`: particle geometry `[dEta, dPhi]` with shape `[num_particles, 2]`.
- `edge_index`: directed k-nearest-neighbour connections in the `(dEta, dPhi)` plane.
- `y`: jet target, where `0` means light quark (UDS) and `1` means gluon.
- `original_flavor`: the original `physFlav` value, retained as metadata only.
- `event_id`: the source event identifier for leakage-safe dataset splitting.

Only particles satisfying `PF_fromAK4Jet == 1` are included as nodes. The edge builder uses native PyTorch, so it does not require `torch-cluster`.

`jet_graphs` is still the complete filtered sample. It must later be split into training, validation, and test subsets by `event_id` before creating PyG `DataLoader` objects.